In [141]:
import pandas as pd
import ast
import nltk
from nltk.stem import  WordNetLemmatizer

In [65]:
movies = pd.read_csv('tmdb_5000_movies.csv')

In [66]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   str    
 2   homepage              1712 non-null   str    
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   str    
 5   original_language     4803 non-null   str    
 6   original_title        4803 non-null   str    
 7   overview              4800 non-null   str    
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   str    
 10  production_countries  4803 non-null   str    
 11  release_date          4802 non-null   str    
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   str    
 15  status                4803 non-n

In [67]:
movies.shape

(4803, 20)

In [68]:
credits = pd.read_csv('tmdb_5000_credits.csv')

In [69]:
credits.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   movie_id  4803 non-null   int64
 1   title     4803 non-null   str  
 2   cast      4803 non-null   str  
 3   crew      4803 non-null   str  
dtypes: int64(1), str(3)
memory usage: 33.8 MB


In [70]:
movies = movies.merge(credits, on = "title")

In [71]:
movies.head

<bound method NDFrame.head of          budget                                             genres  \
0     237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1     300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2     245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3     250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4     260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
...         ...                                                ...   
4804     220000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4805       9000  [{"id": 35, "name": "Comedy"}, {"id": 10749, "...   
4806          0  [{"id": 35, "name": "Comedy"}, {"id": 18, "nam...   
4807          0                                                 []   
4808          0                [{"id": 99, "name": "Documentary"}]   

                                               homepage      id  \
0                           http://www.avatarmovie.com/   1999

In [72]:
movies.shape

(4809, 23)

In [73]:
movies = movies[['title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'tagline']]

In [74]:
movies.shape

(4809, 7)

In [75]:
movies.isnull().sum()

title         0
overview      3
genres        0
keywords      0
cast          0
crew          0
tagline     844
dtype: int64

In [76]:
movies.dropna(inplace=True)

In [77]:
movies.shape

(3965, 7)

In [78]:
movies.duplicated().sum()

np.int64(0)

In [79]:
def convert(text):
   L=[]
   for i in ast.literal_eval(text):
       L.append(i['name'])
   return L

In [80]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [81]:
def convert3(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter !=3:
            L.append(i['name'])
            counter +=1
        else:
            break
    return L

In [82]:
movies['cast'] = movies['cast'].apply(convert3)

In [83]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job']=='Director':
            L.append(i['name'])
            break
    return L

In [84]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [85]:
def collaps(L):
    L1=[]
    for i in L:
        L1.append(i.replace(" ", "_"))
    return L1

In [86]:
movies['cast'] = movies['cast'].apply(collaps)
movies['crew'] = movies['crew'].apply(collaps)
movies['keywords'] = movies['keywords'].apply(collaps)
movies['genres'] = movies['genres'].apply(collaps)

In [87]:
print(movies['genres'].iloc[0])

['Action', 'Adventure', 'Fantasy', 'Science_Fiction']


In [88]:
print(movies['keywords'].iloc[0])

['culture_clash', 'future', 'space_war', 'space_colony', 'society', 'space_travel', 'futuristic', 'romance', 'space', 'alien', 'tribe', 'alien_planet', 'cgi', 'marine', 'soldier', 'battle', 'love_affair', 'anti_war', 'power_relations', 'mind_and_soul', '3d']


In [89]:
print(movies['crew'].iloc[0])

['James_Cameron']


In [98]:
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...


True

In [99]:
lemmatizer = WordNetLemmatizer()

In [100]:
def lemmatize_text(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [101]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [102]:
movies['tagline'] = movies['tagline'].fillna('')

In [103]:
movies['tagline'] = movies['tagline'].apply(lambda x: x.split())

In [104]:
movies['tags'] = (
    movies['overview']
    + movies['genres']
    + movies['keywords']
    + movies['cast']
    + movies['crew']
    + movies['tagline']
)

In [133]:
new_df = movies[[ 'title', 'tags']].copy()

In [134]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

In [135]:
new_df['tags'] = new_df['tags'].apply(lemmatize_text)

In [136]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

In [137]:
new_df.head()

,title,tags
0,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,Spectre,a cryptic message from bond’s past sends him o...
3,The Dark Knight Rises,following the death of district attorney harve...
4,John Carter,"john carter is a war-weary, former military ca..."


In [138]:
new_df.info()

<class 'pandas.DataFrame'>
Index: 3965 entries, 0 to 4807
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   title   3965 non-null   str  
 1   tags    3965 non-null   str  
dtypes: str(2)
memory usage: 2.1 MB


In [139]:
print(type(movies['overview'].iloc[0]))
print(type(movies['tags'].iloc[0]))

<class 'list'>
<class 'list'>


In [140]:
new_df.to_csv("clean_movies.csv", index=False)